# 7.1 - Multi-Provider API Engineering

**Module 7 - LLM API Engineering** | Netsetos GenAI Engineering

This notebook works through Multi-Provider API hands-on: The one call that breaks on Tuesday; Provider 1: Google Gemini; Provider 2: OpenAI (GPT-5.4); Provider 3: Anthropic (Claude); One wrapper, three providers; Automatic failover with retry.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

This first cell is just environment prep - it installs the two SDKs the early demos need and leaves a note about reproducibility. Nothing runs against a provider yet; you're only getting the machine ready.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic openai -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A pure setup cell: an (optional) pip line for the OpenAI and Anthropic SDKs plus a reminder comment. No model is called and no client is built here.

**How the code works - step by step**
- **Commented pip line** - `pip install anthropic openai -q`, left commented so it only fires if you're on a fresh Colab/venv that lacks the SDKs.
- **Reproducibility note** - a reminder that the lesson leans on seeded values so runs are comparable.

**In one line:** install the SDKs, then move on.

**What you'll see:** (no output - environment prep)

## 1 - The one call that breaks on Tuesday

Before building anything, see the problem. This is the naive single-provider call almost everyone ships first: one `client.chat.completions.create`, no try/except, no backup. It works for months - until the provider returns a 503 and there is no Plan B.

> **Needs an OpenAI API key** (set OPENAI_API_KEY).

In [ ]:
# This code worked for six months. Then Tuesday happened.
# Minimal example - deliberately no error handling; this lesson exists to fix that.
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
answer = client.chat.completions.create(
    model="gpt-5.4",
    messages=[{"role": "user", "content": "Summarize today's support tickets."}],
)
print(answer.choices[0].message.content)

# Output: Monday, 9:04 AM -
#   "Today's tickets cluster around three themes: login failures..."
#
# Output: Tuesday, 9:04 AM, during a provider outage -
#   Traceback (most recent call last):
#     ...
#   openai.InternalServerError: Error code: 503 - upstream connect error
#
# Your dashboard: 4,000 users staring at a spinner.
# The status page: "Elevated error rates. Investigating."
# Your code: has no Plan B.

**Explanation**

A deliberately fragile example that exists to be broken. It hits a single OpenAI model directly and prints the answer, with zero error handling - the whole point of the lesson is to fix exactly this failure mode.

**How the code works - step by step**
- **`OpenAI(api_key=...)`** - builds one client bound to a single provider.
- **`client.chat.completions.create(...)`** - sends one user message to `gpt-5.4`.
- **`print(answer.choices[0].message.content)`** - dumps the reply with no guard around it.

**In one line:** one provider, one call, no fallback - fine until it isn't.

**What you'll see:** On a good day, a summary of support tickets. On a provider outage the `# Output:` block shows the alternative reality: an `openai.InternalServerError: 503` traceback and 4,000 users watching a spinner.

## 2 - Provider 1: Google Gemini

Now the three providers one at a time, so you can feel how each SDK differs before hiding those differences behind one wrapper. Gemini is first: note it uses the newer `google-genai` client and passes the system prompt through a `GenerateContentConfig`.

> **Needs a Google API key** (set GOOGLE_API_KEY).

In [ ]:
# =============================================
# PROVIDER 1: Google Gemini
# pip install google-genai  (the legacy google-generativeai SDK
# reached end-of-life 2025-11-30 - use google-genai for new code)
# =============================================

from google import genai
from google.genai import types
import os

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Explain API rate limits in 2 sentences.")
print(f"Gemini: {response.text}")

# With system prompt
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Why do APIs have rate limits?",
    config=types.GenerateContentConfig(
        system_instruction="You are a helpful AI tutor."))
print(f"Gemini (chat): {response.text}")

# Output: (representative run)
# Gemini: A rate limit caps how many requests a client may send per time
# window. Providers use them to protect shared capacity and to bill fairly.
# Gemini (chat): APIs enforce rate limits to keep one heavy user from...

**Explanation**

The first of three parallel demos, each calling one provider raw. This one shows Gemini's shape: a `genai.Client`, `models.generate_content`, and `response.text` - plus how a system instruction is attached via a config object rather than a message role.

**How the code works - step by step**
- **`genai.Client(api_key=...)`** - creates the Gemini client from `GOOGLE_API_KEY`.
- **First `generate_content`** - a plain prompt to `gemini-3.5-flash`; the reply is read straight off `response.text`.
- **Second `generate_content`** - the same call but with `config=types.GenerateContentConfig(system_instruction=...)`, showing that Gemini carries the system prompt in config, not in the messages list.

**In one line:** Gemini's contract - client, `generate_content`, `.text`, and system prompt in config.

**What you'll see:** Two printed lines prefixed `Gemini:` and `Gemini (chat):` - a two-sentence explanation of rate limits, then a second answer shaped by the tutor system prompt.

## 3 - Provider 2: OpenAI (GPT-5.4)

Same question, OpenAI's way. Here the system prompt is a message with `role: system`, the reply lives under `choices[0].message.content`, and token usage comes back on `response.usage`. Watch how different this shape is from Gemini's.

> **Needs an OpenAI API key** (set OPENAI_API_KEY).

In [ ]:
# =============================================
# PROVIDER 2: OpenAI (GPT-5.4)
# Skipping error handling in the side-by-side demos - see production version in Part 3
# pip install openai
# =============================================

import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.chat.completions.create(
    model="gpt-5.4",
    messages=[
        {"role": "system", "content": "You are a helpful AI tutor."},
        {"role": "user", "content": "Explain API rate limits in 2 sentences."},
    ],
    temperature=0.3,
)

print(f"OpenAI: {response.choices[0].message.content}")
print(f"Tokens: {response.usage.total_tokens}")

# Output: (representative run)
# OpenAI: Rate limits cap how many requests or tokens you can send per
# minute. They protect the provider's infrastructure and your bill.
# Tokens: 87

**Explanation**

The second raw demo, showing OpenAI's chat-completions contract. The contrast with cell 2 is the lesson: OpenAI puts the system prompt in the `messages` list and exposes usage as `usage.total_tokens`.

**How the code works - step by step**
- **`OpenAI(api_key=...)`** - builds the OpenAI client.
- **`chat.completions.create(...)`** - sends a `messages` list where the system prompt is its own `{'role': 'system'}` entry, with `temperature=0.3`.
- **`response.choices[0].message.content`** - the answer path, printed with the token count from `response.usage.total_tokens`.

**In one line:** OpenAI's contract - `messages` list with a system role, answer under `choices[0].message`.

**What you'll see:** Two lines: `OpenAI:` with a two-sentence rate-limit answer, and `Tokens:` with the total token count (~87 in the sample run).

## 4 - Provider 3: Anthropic (Claude)

The third shape. Claude uses `messages.create`, the system prompt is a top-level `system=` argument, the reply is a content block at `content[0].text`, and usage splits into `input_tokens` and `output_tokens`. Crucially, Claude *requires* `max_tokens` - the others default it.

> **Needs an Anthropic API key** (set ANTHROPIC_API_KEY).

In [ ]:
# =============================================
# PROVIDER 3: Anthropic (Claude)
# Skipping error handling in the side-by-side demos - see production version in Part 3
# pip install anthropic
# =============================================

import os
from anthropic import Anthropic

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    system="You are a helpful AI tutor.",
    messages=[
        {"role": "user", "content": "Explain API rate limits in 2 sentences."},
    ],
)

print(f"Claude: {response.content[0].text}")
print(f"Tokens: {response.usage.input_tokens + response.usage.output_tokens}")

# Output: (representative run)
# Claude: A rate limit is a ceiling on requests or tokens per time window,
# enforced so shared capacity stays fair and predictable for every client.
# Tokens: 102
# Note the asymmetry: Anthropic REQUIRES max_tokens; OpenAI and Gemini default it.

**Explanation**

The last raw demo, exposing Anthropic's contract. It surfaces the one asymmetry worth remembering: `max_tokens` is mandatory here, and token usage arrives as two separate fields you must add together.

**How the code works - step by step**
- **`Anthropic(api_key=...)`** - builds the Claude client.
- **`messages.create(...)`** - passes `system=` as its own keyword and a required `max_tokens=1024`.
- **`response.content[0].text`** - Claude returns a list of content blocks; the text is inside the first one.
- **`usage.input_tokens + usage.output_tokens`** - total tokens are summed by hand from the two fields.

**In one line:** Claude's contract - top-level `system`, required `max_tokens`, answer in `content[0].text`.

**What you'll see:** Two lines: `Claude:` with the rate-limit answer and `Tokens:` with the summed total (~102 in the sample). The trailing comment flags the `max_tokens` asymmetry.

## 5 - One wrapper, three providers

Three SDKs, three shapes - now hide all of it. `UnifiedLLM` is the first real building block: one `chat()` method that detects the provider from the model name, translates to that SDK's contract, and returns the same `LLMResponse` dataclass every time. Swapping providers becomes changing one string.

In [ ]:
# =============================================
# UNIFIED LLM WRAPPER
# One interface. Three providers.
# Swap providers by changing one string.
# =============================================

import os, time
from dataclasses import dataclass

@dataclass
class LLMResponse:
    """Unified response format - same regardless of provider."""
    text: str
    provider: str
    model: str
    input_tokens: int
    output_tokens: int
    latency_ms: float
    cost_usd: float

class UnifiedLLM:
    """Call any LLM provider through one consistent interface."""
    
    # Pricing per 1M tokens (approximate, 2026)
    PRICING = {
        "gemini-3.5-flash":          {"input": 1.50,  "output": 9.00},
        "gemini-3.1-flash-lite":     {"input": 0.25,  "output": 1.50},
        "gemini-2.5-pro":            {"input": 1.25,  "output": 10.00},
        "gpt-5.4":                   {"input": 2.50,  "output": 15.00},
        "gpt-5.4-mini":               {"input": 0.75,  "output": 4.50},
        "claude-sonnet-4-6":  {"input": 3.00,  "output": 15.00},
        "claude-haiku-4-5":          {"input": 1.00,  "output": 5.00},
    }
    
    def __init__(self):
        self._clients = {}
        self._init_providers()
    
    def _init_providers(self):
        # Initialize available providers (only if API key exists)
        if os.getenv("GOOGLE_API_KEY"):
            from google import genai
            self._clients["gemini"] = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
        
        if os.getenv("OPENAI_API_KEY"):
            from openai import OpenAI
            self._clients["openai"] = OpenAI()
        
        if os.getenv("ANTHROPIC_API_KEY"):
            from anthropic import Anthropic
            self._clients["anthropic"] = Anthropic()
        
        print(f"Initialized providers: {list(self._clients.keys())}")
    
    def _calc_cost(self, model, input_tok, output_tok):
        pricing = self.PRICING.get(model, {"input": 0, "output": 0})
        return (input_tok * pricing["input"] + output_tok * pricing["output"]) / 1_000_000
    
    def _detect_provider(self, model):
        if "gemini" in model: return "gemini"
        if "gpt" in model: return "openai"
        if "claude" in model: return "anthropic"
        raise ValueError(f"Unknown model: {model}")
    
    def chat(self, message: str, model: str = "gemini-3.5-flash",
            system: str = "", temperature: float = 0.3) -> LLMResponse:
        """Send a message to any provider. Returns unified response."""
        provider = self._detect_provider(model)
        start = time.time()

        if provider == "gemini":
            from google.genai import types
            resp = self._clients["gemini"].models.generate_content(
                model=model, contents=message,
                config=types.GenerateContentConfig(
                    system_instruction=system or None, temperature=temperature))
            text = resp.text
            in_tok = resp.usage_metadata.prompt_token_count
            out_tok = resp.usage_metadata.candidates_token_count
        
        elif provider == "openai":
            msgs = []
            if system: msgs.append({"role": "system", "content": system})
            msgs.append({"role": "user", "content": message})
            resp = self._clients["openai"].chat.completions.create(
                model=model, messages=msgs, temperature=temperature)
            text = resp.choices[0].message.content
            in_tok = resp.usage.prompt_tokens
            out_tok = resp.usage.completion_tokens
        
        elif provider == "anthropic":
            resp = self._clients["anthropic"].messages.create(
                model=model, max_tokens=2048,
                system=system or "You are a helpful assistant.",
                messages=[{"role": "user", "content": message}],
                temperature=temperature)
            text = resp.content[0].text
            in_tok = resp.usage.input_tokens
            out_tok = resp.usage.output_tokens
        
        latency = (time.time() - start) * 1000
        cost = self._calc_cost(model, in_tok, out_tok)
        
        return LLMResponse(text=text, provider=provider, model=model,
                          input_tokens=in_tok, output_tokens=out_tok,
                          latency_ms=latency, cost_usd=cost)

# ── Use it! Same code, any provider ──
llm = UnifiedLLM()

# Switch providers by changing one string:
for model in ["gemini-3.5-flash", "gpt-5.4-mini", "claude-haiku-4-5"]:
    resp = llm.chat("What is RAG in one sentence?", model=model)
    print(f"  [{resp.provider:9s}] {resp.latency_ms:6.0f}ms  ${resp.cost_usd:.5f}  {resp.text[:80]}")

# Output: (representative run)
# Initialized providers: ['gemini', 'openai', 'anthropic']
#   [gemini   ]    412ms  $0.00036  A rate limit caps how many requests a client...
#   [openai   ]    633ms  $0.00018  Rate limits protect shared infrastructure and...
#   [anthropic]    587ms  $0.00020  A rate limit is a ceiling on requests or tok...

**Explanation**

The abstraction layer that makes everything after it possible. It normalizes the three different SDK shapes from cells 2-4 into one call and one return type, and it bolts on cost and latency measurement so every response is comparable. Read it as: detect provider from model name, dispatch to the right SDK, always hand back the same struct.

**How the code works - step by step**
- **`LLMResponse` dataclass** - the single return shape: text, provider, model, token counts, latency, and computed cost.
- **`PRICING` table** - per-1M-token input/output rates used to price each call.
- **`_init_providers`** - lazily builds a client for each provider *only if* its API key is present, so the class runs with whatever keys you have.
- **`_detect_provider`** - maps a model string to a provider by substring (`gemini`/`gpt`/`claude`).
- **`_calc_cost`** - turns token counts into dollars from the pricing table.
- **`chat(...)`** - the one public method: branches per provider to call the right SDK, extracts text and token counts, times the call, and packs it all into an `LLMResponse`.
- **Bottom loop** - calls the same `chat()` across three models, proving the swap is a one-string change.

**In one line:** one method in, one `LLMResponse` out - the provider is just a string.

**What you'll see:** First `Initialized providers: [...]` listing whichever keys were found, then one line per model showing provider, latency in ms, cost, and a truncated answer.

## 6 - Automatic failover with retry

One wrapper still dies if its single provider is down. `FallbackRouter` fixes the Tuesday problem: it walks a chain of models, retries each with exponential backoff, and moves to the next provider when one keeps failing - so a 503 on the primary silently becomes a success on the backup.

In [ ]:
# =============================================
# FALLBACK ROUTER: Automatic failover + retry
# Try primary → fallback 1 → fallback 2
# with exponential backoff retry per provider.
# =============================================

import time, logging
from collections import defaultdict

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("llm_router")

class FallbackRouter:
    """Route requests with automatic fallback and retry."""
    
    def __init__(self, llm: UnifiedLLM,
                 chain: list[str] = None,
                 max_retries: int = 3,
                 timeout_ms: float = 15000):
        self.llm = llm
        self.chain = chain or ["gpt-5.4", "claude-sonnet-4-6", "gemini-3.5-flash"]
        self.max_retries = max_retries
        self.timeout_ms = timeout_ms
        # defaultdict, not a fixed dict: SmartRouter swaps the chain later,
        # and stats must not KeyError on models it has never seen
        self.stats = defaultdict(lambda: {"success": 0, "fail": 0})
    
    def _try_with_retry(self, message, model, system, temperature):
        """Try a model with exponential backoff retry."""
        for attempt in range(self.max_retries):
            try:
                resp = self.llm.chat(message, model=model,
                                    system=system, temperature=temperature)
                
                # Check latency threshold
                if resp.latency_ms > self.timeout_ms:
                    log.warning(f"{model}: too slow ({resp.latency_ms:.0f}ms)")
                    raise TimeoutError("Latency exceeded threshold")
                
                self.stats[model]["success"] += 1
                return resp
            
            except Exception as e:
                log.warning(f"{model} attempt {attempt+1} failed: {e}")
                if attempt < self.max_retries - 1:  # never sleep after the LAST attempt -
                    wait = 2 ** attempt             # that only delays the failover
                    log.warning(f"Retry in {wait}s")
                    time.sleep(wait)
        
        self.stats[model]["fail"] += 1
        return None
    
    def route(self, message: str, system: str = "",
             temperature: float = 0.3) -> LLMResponse:
        """Try each model in the chain until one succeeds."""
        
        for model in self.chain:
            log.info(f"Trying: {model}")
            resp = self._try_with_retry(message, model, system, temperature)
            
            if resp:
                if model != self.chain[0]:
                    log.info(f"Used fallback: {model}")
                return resp
        
        raise RuntimeError("All providers failed!")
    
    def health_report(self):
        print("\nProvider Health Report:")
        for model, stats in self.stats.items():
            total = stats["success"] + stats["fail"]
            rate = stats["success"] / total * 100 if total > 0 else 0
            status = "🟢" if rate > 90 else "🟡" if rate > 50 else "🔴"
            print(f"  {status} {model:30s}  {stats['success']}/{total} ({rate:.0f}%)")

# ── Use it ──
router = FallbackRouter(llm)

# This automatically falls back if the primary fails
response = router.route(
    "What are the benefits of vector databases for RAG?",
    system="You are a senior AI engineer. Be concise.",
)

print(f"\nProvider used: {response.provider}")
print(f"Latency: {response.latency_ms:.0f}ms")
print(f"Cost: ${response.cost_usd:.5f}")
print(f"Answer: {response.text[:150]}...")

router.health_report()

# Output: (representative run, primary forced down to show the failover)
# INFO:llm_router:Trying: gpt-5.4
# WARNING:llm_router:gpt-5.4 attempt 1 failed: 503 Service Unavailable
# WARNING:llm_router:Retry in 1s
# WARNING:llm_router:gpt-5.4 attempt 2 failed: 503 Service Unavailable
# WARNING:llm_router:Retry in 2s
# WARNING:llm_router:gpt-5.4 attempt 3 failed: 503 Service Unavailable
# INFO:llm_router:Trying: claude-sonnet-4-6
# INFO:llm_router:Used fallback: claude-sonnet-4-6
#
# Provider used: anthropic
# Latency: 641ms   Cost: $0.00209
# Answer: Retry a failed request when the error is transient - a 429 or 5xx...
#
# Provider Health Report:
#   🔴 gpt-5.4                0/1 (0%)
#   🟢 claude-sonnet-4-6      1/1 (100%)

**Explanation**

The reliability layer built on top of UnifiedLLM. It turns 'one call that can fail' into 'a chain that keeps trying until something works', with per-attempt backoff, a latency ceiling that treats slow as failed, and running success/fail stats per model. This is the piece that removes the single point of failure from cell 1.

**How the code works - step by step**
- **`__init__`** - stores the LLM, a default fallback `chain`, retry count, and a latency `timeout_ms`; stats are a `defaultdict` so a later swapped-in chain never KeyErrors.
- **`_try_with_retry`** - loops up to `max_retries`: calls the model, raises if latency exceeds the threshold, sleeps `2 ** attempt` seconds between tries, and deliberately never sleeps after the final attempt (that would only delay failover).
- **`route`** - walks the chain in order, returns the first success, and logs when a fallback (not the primary) was the one that answered; raises `RuntimeError` only if every provider fails.
- **`health_report`** - prints per-model success rate with a 🟢/🟡/🔴 status.

**In one line:** try each model with backoff, fall through to the next, report who's healthy.

**What you'll see:** Log lines showing the primary (`gpt-5.4`) failing three times with retry waits, then the fallback (`claude-sonnet-4-6`) succeeding - followed by the answer, its provider/latency/cost, and a health report with colored status per model.

## 7 - Send each task to its best model

Failover keeps you alive; smart routing keeps you cheap and good. `SmartRouter` classifies each prompt (code, reasoning, long-doc, creative, structured, simple) with a cheap keyword heuristic, then points it at the model that's best - and cheapest-appropriate - for that job, reusing the FallbackRouter for reliability underneath.

In [ ]:
# =============================================
# SMART ROUTER: Send each task to the BEST model
# for that specific task type.
# =============================================

class SmartRouter:
    """Route tasks to the best provider based on task type."""
    
    # Which model is best at what?
    ROUTING_TABLE = {
        "code":         {"primary": "gpt-5.4",               "fallback": "gemini-2.5-pro"},
        "long_doc":     {"primary": "gemini-3.5-flash",      "fallback": "claude-sonnet-4-6"},
        "reasoning":    {"primary": "claude-sonnet-4-6",    "fallback": "gpt-5.4"},
        "simple":       {"primary": "gemini-3.1-flash-lite", "fallback": "gpt-5.4-mini"},
        "creative":     {"primary": "claude-sonnet-4-6",    "fallback": "gpt-5.4"},
        "structured":   {"primary": "gemini-3.1-flash-lite", "fallback": "gpt-5.4-mini"},
    }
    
    def __init__(self, llm: UnifiedLLM):
        self.llm = llm
        self.fallback_router = FallbackRouter(llm)
    
    def classify_task(self, message: str) -> str:
        """Detect what kind of task this is (cheap heuristic)."""
        msg = message.lower()
        
        if any(w in msg for w in ["code", "function", "debug", "python", "javascript", "api", "error"]):
            return "code"
        if any(w in msg for w in ["summarize", "document", "analyze this", "read this"]):
            return "long_doc"
        if any(w in msg for w in ["think", "reason", "pros and cons", "compare", "ethics"]):
            return "reasoning"
        if any(w in msg for w in ["write", "story", "creative", "poem", "essay"]):
            return "creative"
        if any(w in msg for w in ["json", "extract", "classify", "format"]):
            return "structured"
        return "simple"
    
    def route(self, message: str, system: str = "", **kwargs) -> LLMResponse:
        """Automatically route to the best model for this task."""
        task = self.classify_task(message)
        routing = self.ROUTING_TABLE[task]
        
        chain = [routing["primary"], routing["fallback"]]
        self.fallback_router.chain = chain
        
        resp = self.fallback_router.route(message, system=system, **kwargs)
        print(f"  Task: {task:12s} → Model: {resp.model:30s} (${resp.cost_usd:.5f})")
        return resp

# ── Test with different task types ──
smart = SmartRouter(llm)

tasks = [
    "Write a Python function that sorts a list using merge sort.",
    "Summarize this 50-page document about climate change.",
    "Compare the pros and cons of microservices vs monolith architecture.",
    "Write a short story about a robot discovering music.",
    "Extract product name and price from this review as JSON.",
    "What is the capital of France?",
]

print("Smart routing results:\n")
for task in tasks:
    smart.route(task)

# Output: (representative run)
# Smart routing results:
#   Task: code         → Model: gpt-5.4                   ($0.00306)
#   Task: long_doc     → Model: gemini-3.5-flash          ($0.00144)
#   Task: reasoning    → Model: claude-sonnet-4-6         ($0.00224)
#   Task: creative     → Model: claude-sonnet-4-6         ($0.00205)
#   Task: structured   → Model: gemini-3.1-flash-lite     ($0.00008)
#   Task: simple       → Model: gemini-3.1-flash-lite     ($0.00004)
# Note the ~77x cost spread between the cheapest and priciest route -
# that spread is the whole business case for routing (animation below).

**Explanation**

The routing layer that sits on top of the fallback router. Instead of one fixed chain, it swaps in a task-specific primary+fallback pair per request, so a trivial question goes to a cheap flash model while a coding task goes to a strong one. The payoff is a large cost spread across task types for the same code path.

**How the code works - step by step**
- **`ROUTING_TABLE`** - maps each task type to a `{primary, fallback}` model pair.
- **`__init__`** - holds the LLM and its own `FallbackRouter` instance to do the actual calling.
- **`classify_task`** - a cheap heuristic: lowercases the message and keyword-matches to a task type, defaulting to `simple`.
- **`route`** - classifies, looks up the pair, overwrites the fallback router's `chain` with `[primary, fallback]`, delegates the call, and prints which task type mapped to which model at what cost.
- **Bottom loop** - runs six varied prompts to show each landing on a different model.

**In one line:** classify the prompt, pick the right primary+fallback, let the fallback router run it.

**What you'll see:** A `Smart routing results` block with one line per task showing task type → chosen model and its cost. The trailing comment highlights the ~77x cost spread between the cheapest and priciest route - the business case for routing.

## 8 - Watch the spend in real time

Routing to cheap models only matters if you can prove it. `CostTracker` records every call, sums spend per provider and per day, warns when you cross 80% of a daily budget, and prints a spend report - turning invisible token costs into a dashboard.

In [ ]:
# =============================================
# COST TRACKER: Real-time spend monitoring
# =============================================

from collections import defaultdict
from datetime import datetime

class CostTracker:
    """Track LLM spending per provider, model, and time period."""
    
    def __init__(self, daily_budget_usd: float = 10.0):
        self.daily_budget = daily_budget_usd
        self.records = []
    
    def log(self, response: LLMResponse):
        self.records.append({
            "timestamp": datetime.now(),
            "provider": response.provider,
            "model": response.model,
            "input_tokens": response.input_tokens,
            "output_tokens": response.output_tokens,
            "cost": response.cost_usd,
            "latency_ms": response.latency_ms,
        })
        
        # Budget alert
        today_cost = self.today_spend()
        if today_cost > self.daily_budget * 0.8:
            print(f"  ⚠️ BUDGET WARNING: ${today_cost:.2f} / ${self.daily_budget:.2f} today")
    
    def today_spend(self):
        today = datetime.now().date()
        return sum(r["cost"] for r in self.records if r["timestamp"].date() == today)
    
    def report(self):
        print("\n💰 Cost Report")
        print("─" * 50)
        
        # Per provider
        by_provider = defaultdict(lambda: {"cost": 0, "calls": 0, "tokens": 0})
        for r in self.records:
            p = by_provider[r["provider"]]
            p["cost"] += r["cost"]
            p["calls"] += 1
            p["tokens"] += r["input_tokens"] + r["output_tokens"]
        
        print(f"\n{'Provider':<12} {'Calls':>6} {'Tokens':>10} {'Cost':>10} {'Avg/call':>10}")
        total_cost = 0
        for prov, data in sorted(by_provider.items()):
            avg = data["cost"] / data["calls"] if data["calls"] else 0
            print(f"  {prov:<12} {data['calls']:>4} {data['tokens']:>8,} ${data['cost']:>8.4f} ${avg:>8.5f}")
            total_cost += data["cost"]
        
        print(f"\n  Total: ${total_cost:.4f}  |  Today: ${self.today_spend():.4f} / ${self.daily_budget:.2f}")

# ── Wire it up with the router ──
tracker = CostTracker(daily_budget_usd=5.0)

for task in tasks:
    resp = smart.route(task)
    tracker.log(resp)

tracker.report()

# Output: (representative run)
# 💰 Cost Report
# ──────────────────────────────────────────────────
# Provider      Calls     Tokens       Cost   Avg/call
#   anthropic       2      1,842    $0.0043   $0.00215
#   gemini          3      1,201    $0.0016   $0.00053
#   openai          1        698    $0.0031   $0.00306
#
#   Total: $0.0089  |  Today: $0.0089 / $5.00

**Explanation**

The observability layer. It doesn't call any model - it consumes the `LLMResponse` objects the other layers produce and aggregates them into per-provider and per-day totals with a budget alarm. Think of it as the accountant that sits beside the router.

**How the code works - step by step**
- **`__init__`** - sets a daily budget and an empty `records` list.
- **`log`** - appends one record (timestamp, provider, model, tokens, cost, latency) per response, then prints a ⚠️ warning if today's spend passes 80% of budget.
- **`today_spend`** - sums costs whose timestamp is today.
- **`report`** - groups records by provider and prints a table of calls, tokens, cost, and average cost per call, plus grand total and today-vs-budget.

**In one line:** log every call, alert near budget, print a per-provider spend table.

**What you'll see:** A `💰 Cost Report` table with one row per provider (calls, tokens, cost, avg/call) and a total line showing today's spend against the $5.00 budget.

## 9 - Everything wired into one class

The four building blocks - UnifiedLLM, SmartRouter, CostTracker - collapse into `ProductionLLM`, exposing a single `.ask()` method. Pass no model and it auto-routes; pass a model and it uses that one first with the rest as fallbacks. Budget is checked before every call. This is the drop-in you'd actually put in an app.

In [ ]:
# =============================================
# PRODUCTION MULTI-PROVIDER SYSTEM
# All components wired together.
# Drop this into any Python app.
# =============================================

class ProductionLLM:
    """Production-grade LLM client with multi-provider support."""
    
    def __init__(self, daily_budget: float = 10.0):
        self.llm = UnifiedLLM()
        self.router = SmartRouter(self.llm)
        self.tracker = CostTracker(daily_budget_usd=daily_budget)
    
    def ask(self, message: str, system: str = "",
           model: str = None, temperature: float = 0.3) -> LLMResponse:
        """
        The ONE method you use in your app.
        
        - model=None → auto-route to best model for the task
        - model="gpt-5.4" → use specific model with fallback
        """
        
        # Check budget BEFORE making the call
        if self.tracker.today_spend() > self.tracker.daily_budget:
            raise RuntimeError("Daily budget exceeded! Call tracker.report() for details.")
        
        # Route the request
        if model:
            # Specific model requested: put it FIRST in the chain so it is
            # actually used, keeping the others as fallbacks behind it
            fr = self.router.fallback_router
            fr.chain = [model] + [m for m in fr.chain if m != model]
            resp = fr.route(message, system=system, temperature=temperature)
        else:
            # Auto-route to best model for this task
            resp = self.router.route(message, system=system, temperature=temperature)
        
        # Track cost
        self.tracker.log(resp)
        return resp
    
    def report(self):
        self.tracker.report()
        self.router.fallback_router.health_report()

# ── Usage in your app ──
ai = ProductionLLM(daily_budget=5.0)

# Auto-route (default - best model picked automatically)
resp = ai.ask("Write a Python function to calculate compound interest.")
print(f"Answer: {resp.text[:100]}...")

# Specific model (with automatic fallback if it fails)
resp = ai.ask("What is the meaning of life?", model="gemini-3.5-flash")
print(f"Answer: {resp.text[:100]}...")

# With system prompt
resp = ai.ask(
    "Explain transformers to a 10-year-old.",
    system="You are a friendly teacher who uses Indian cricket analogies."
)
print(f"Answer: {resp.text[:100]}...")

# Get the full dashboard
ai.report()

print("""
What ProductionLLM gives you:
  ✅ One .ask() method for everything
  ✅ Auto-routing to the best model per task type
  ✅ Automatic fallback if any provider goes down
  ✅ Retry with exponential backoff
  ✅ Latency monitoring (skips slow providers)
  ✅ Real-time cost tracking with budget alerts
  ✅ Health dashboard showing reliability per provider
  
This is what production AI applications use internally.
""")

# Output: (representative run)
#   Task: code         → Model: gpt-5.4                   ($0.00299)
# Answer: def compound_interest(principal: float, rate: float, years: int)...
# Answer: The meaning of life is a question philosophy has debated for mill...
# Answer: Imagine your cricket team has a coach who watched every match eve...
# (then tracker.report() and health_report() print the dashboards from Part 5)

**Explanation**

The integration cell that composes all prior layers into one object with one method. It adds two real production behaviors on top: a hard budget gate that refuses calls once you're over, and a model-override path that puts your chosen model at the front of the fallback chain instead of ignoring the fallbacks.

**How the code works - step by step**
- **`__init__`** - constructs the UnifiedLLM, wraps it in a SmartRouter, and attaches a CostTracker.
- **`ask(...)`** - the one public method: first raises if today's spend already exceeds the budget; if a `model` is given it rebuilds the fallback chain as `[model] + others` so your pick leads but backups remain; otherwise it auto-routes by task; then it logs the cost and returns the response.
- **`report`** - prints both the cost report and the provider health report.
- **Bottom usage block** - shows auto-route, forced-model, and system-prompt calls, then prints a feature checklist of what the class buys you.

**In one line:** one `.ask()` - auto-route or override, always budgeted, always tracked.

**What you'll see:** Truncated answers for the three example asks, then the combined cost and health dashboards, and a printed checklist of what ProductionLLM provides (auto-routing, fallback, retry, latency skipping, cost tracking, health).

## 10 - Pick a 2026 provider by need, not brand

The rest of the notebook zooms out to the wider landscape. This cell is a plain lookup table: match the job's dominant need - raw speed, low cost, EU data residency, coding, or huge context - to the provider that leads on it, rather than defaulting to one brand for everything.

In [ ]:
# Pick a 2026 provider by the job's dominant need, not by brand loyalty.
NEED_TO_MODEL = {
    "speed":    "groq/llama-3.3-70b",        # LPU hardware, fastest tok/s
    "cheap":    "deepseek/deepseek-v4-flash", # much cheaper than frontier
    "eu":       "mistral/mistral-large",      # EU sovereignty + GDPR
    "coding":   "anthropic/claude-sonnet-4-6",
    "longdoc":  "gemini/gemini-2.5-pro",      # 1M-token context
}

def pick(need):
    return NEED_TO_MODEL.get(need, "openai/gpt-5.4")  # general-purpose default

for need in ["speed", "cheap", "coding"]:
    print(f"{need:7s} -> {pick(need)}")

# Output:
#   speed   -> groq/llama-3.3-70b
#   cheap   -> deepseek/deepseek-v4-flash
#   coding  -> anthropic/claude-sonnet-4-6

**Explanation**

A reference/decision-table cell, not a model call. It encodes the 2026 'which provider wins at what' map as a dict plus a tiny `pick` function with a general-purpose default, so provider choice becomes a data lookup.

**How the code works - step by step**
- **`NEED_TO_MODEL`** - maps a need (`speed`/`cheap`/`eu`/`coding`/`longdoc`) to the standout model for it, each annotated with why (LPU hardware, cheapest, GDPR, etc.).
- **`pick(need)`** - returns the mapped model, falling back to a general-purpose OpenAI default for unknown needs.
- **Bottom loop** - prints the pick for three sample needs.

**In one line:** name the dominant need, get the provider that leads on it.

**What you'll see:** Three lines mapping `speed`, `cheap`, and `coding` to their recommended provider/model strings.

## 11 - The LiteLLM shortcut

Writing per-provider branches (as UnifiedLLM did by hand) is one option; LiteLLM is the library that does it for you. One `completion()` call speaks every provider's dialect - you change only the model string, and keys are read from environment variables.

> **Needs the relevant provider API keys** in env vars (OpenAI, Anthropic, Groq, DeepSeek).

In [ ]:
# One completion() call across every provider - only the model string changes.
# pip install litellm
from litellm import completion

MODELS = [
    "gpt-5.4",                       # OpenAI
    "anthropic/claude-sonnet-4-6",   # Anthropic
    "groq/llama-3.3-70b",            # Groq
    "deepseek/deepseek-v4-flash",    # DeepSeek
]
msgs = [{"role": "user", "content": "Name one benefit of vector databases."}]

for model in MODELS:
    try:
        resp = completion(model=model, messages=msgs, timeout=30)  # keys from env vars
        print(f"{model:30s} -> {resp.choices[0].message.content[:40]}")
    except Exception as e:
        print(f"{model:30s} -> FAILED ({type(e).__name__}); router would fall back")

# Output: (representative)
#   gpt-5.4                        -> Vector databases enable fast seman...
#   groq/llama-3.3-70b             -> Sub-second nearest-neighbour searc...

**Explanation**

A shortcut cell showing the off-the-shelf alternative to the hand-rolled wrapper. `litellm.completion` normalizes the call and response across providers, so the same loop hits four different vendors with only the model string changing - and a try/except shows where a router would fall back.

**How the code works - step by step**
- **`from litellm import completion`** - the single unified entry point.
- **`MODELS` list** - four provider-prefixed model strings (OpenAI, Anthropic, Groq, DeepSeek).
- **Loop with try/except** - calls `completion(model=..., messages=..., timeout=30)` for each; keys come from env vars, and a failure prints a `FAILED` line noting a router would fall back.

**In one line:** one `completion()` call, every provider, only the model string changes.

**What you'll see:** One line per model showing the model name and a truncated answer, or a `FAILED (ErrorName)` line for any provider whose key is missing or that errors.

## 12 - Fan out a batch concurrently

When you have many non-urgent prompts, calling them one by one wastes time. This cell fires them concurrently with `asyncio`, but caps in-flight requests with a semaphore so you respect rate limits, and swallows per-row errors so one bad row can't sink the whole batch.

> **Needs an OpenAI API key** (set OPENAI_API_KEY).

In [ ]:
# Fan out non-urgent prompts concurrently, capped to respect rate limits.
import asyncio
from openai import AsyncOpenAI

client = AsyncOpenAI()
sem = asyncio.Semaphore(10)  # at most 10 requests in flight

async def classify(text):
    async with sem:
        try:
            resp = await client.chat.completions.create(
                model="gpt-5.4-mini", timeout=30,
                messages=[{"role": "user", "content": f"Label sentiment: {text}"}])
            return resp.choices[0].message.content
        except Exception:
            return "error"  # one bad row must not sink the batch

async def main(rows):
    return await asyncio.gather(*(classify(r) for r in rows))

# For thousands of rows with no latency need, submit the SAME prompts via the
# Batch API instead: JSONL upload, about 50 percent off, results within 24h.

# Output:
#   ['positive', 'negative', 'neutral', ...]

**Explanation**

A throughput pattern cell. It shows the safe way to parallelize LLM calls: bounded concurrency via a semaphore plus per-task error isolation, with a comment pointing to the cheaper Batch API for truly large, latency-insensitive jobs.

**How the code works - step by step**
- **`AsyncOpenAI()` + `Semaphore(10)`** - an async client and a cap of at most 10 concurrent requests.
- **`classify(text)`** - acquires the semaphore, makes one async completion, and returns `'error'` on any exception so a single failure doesn't crash the gather.
- **`main(rows)`** - `asyncio.gather` over all rows to run them concurrently under the cap.
- **Trailing comment** - notes that for thousands of rows the Batch API (JSONL upload, ~50% off, results within 24h) is the better tool.

**In one line:** gather many calls concurrently, capped by a semaphore, errors contained per row.

**What you'll see:** A list of per-row labels such as `['positive', 'negative', 'neutral', ...]`, with `'error'` in any slot whose call failed.

## 13 - Region-pinned calls with PII masked first

The final cell handles a compliance reality: some data can't leave a jurisdiction, and PII shouldn't travel in the clear. It masks Aadhaar numbers before the call and routes through a Bedrock endpoint in the Mumbai region so processing stays inside India for DPDPA.

> **Needs AWS credentials** with Bedrock access in ap-south-1.

In [ ]:
# Route Aadhaar-linked text to an India-region endpoint, masked first.
import re, boto3

def mask_aadhaar(text):
    # Keep only the last 4 digits of any 12-digit Aadhaar number.
    return re.sub(r"\b\d{8}(\d{4})\b", r"XXXXXXXX\1", text)

# Bedrock in Mumbai (ap-south-1) keeps processing inside India for DPDPA.
from botocore.config import Config
cfg = Config(region_name="ap-south-1", read_timeout=30, retries={"max_attempts": 2})
client = boto3.client("bedrock-runtime", config=cfg)

def ask_in_region(user_text):
    safe = mask_aadhaar(user_text)  # PII never leaves in the clear
    try:
        resp = client.converse(modelId="anthropic.claude-sonnet-4-6",
            messages=[{"role": "user", "content": [{"text": safe}]}])
        return resp["output"]["message"]["content"][0]["text"]
    except Exception as e:
        return f"region call failed: {e}"  # fail safe, do not leak PII in logs

print(mask_aadhaar("My Aadhaar is 123456789012"))

# Output:
#   My Aadhaar is XXXXXXXX9012

**Explanation**

A data-governance pattern cell. It combines two ideas: regex-mask sensitive identifiers before anything is sent, and pin the model call to an in-country region via Bedrock so data residency rules are met. The error path fails safe by never echoing PII into logs.

**How the code works - step by step**
- **`mask_aadhaar(text)`** - a regex that keeps only the last 4 digits of any 12-digit Aadhaar number, replacing the first 8 with `X`.
- **Bedrock client with `Config(region_name='ap-south-1', ...)`** - pins the call to Mumbai with a read timeout and retry cap.
- **`ask_in_region(user_text)`** - masks first, then calls `client.converse` on a Claude model in-region; on failure returns a generic error rather than leaking the input.
- **`print(mask_aadhaar(...))`** - a quick local demo of the masking, no cloud call needed to see it work.

**In one line:** mask the PII, then call a model pinned to an in-country region, failing safe.

**What you'll see:** One line from the masking demo: `My Aadhaar is XXXXXXXX9012` - the first eight digits hidden, the last four kept.

You started with one API call that dies the moment its provider has a bad Tuesday, and you finished with `ProductionLLM.ask()` - a single method that auto-picks the right model for each task, fails over to a backup when a provider goes down, retries with backoff, watches latency, and tracks spend against a budget. The four building blocks (UnifiedLLM, FallbackRouter, SmartRouter, CostTracker) each solve one problem and then compose into that one class. The closing cells widen the lens to the 2026 provider landscape, the LiteLLM shortcut, concurrent batching, and region-pinned PII handling. Next, Module 7.2 builds on this to add streaming, structured outputs, and token-level cost controls on top of the multi-provider foundation.